# Agent 与 Vast 协同分析流程

系统通过 **Agent 建模、checker 分析、Agent 验证** 三个阶段完成资源问题检测。

函数建模主要面向**库函数和函数指针调用**：Agent 阅读实现与调用上下文，把函数对资源的影响整理成摘要。checker 在调用处直接应用摘要，减少重复进入函数实现的开销，并补充间接调用所需的语义。

函数摘要回答四个问题：**分配了什么、释放了什么、做了哪些调用方可见的修改，以及资源由谁拥有。** checker 通过摘要追踪资源的生命周期，检查泄漏、释放后使用和重复释放。


## 1. 整体架构

```mermaid
flowchart LR
    A["构建 IR<br/>生成 .thin"] --> B["Agent 建模"]
    B --> C["plugin 写入数据库"]
    C --> D["checker 分析"]
    D --> E["Agent 验证报错"]
    E --> F["tp.json / fp.json"]
    E -->|证据不足| G["询问用户"]
    G --> E
```

系统包含三个主要模块：

| 模块 | 输入 | 职责与输出 |
| --- | --- | --- |
| Agent 建模服务 | 函数实现、类型、调用关系、调用点 | 生成带源码依据的函数摘要，通过 plugin 写入分析数据库 |
| vast-checker | `.thin`、分析数据库 | 查询数据库中的函数模型与 callee 结果，分析程序并写回结果 |
| Agent 验证服务 | 告警、源码、相关摘要与函数结果 | 验证 checker 的报错是否成立，输出 tp.json 和 fp.json |

系统使用统一的分析数据库，组织形式类似 `.git`。plugin 将 Agent 生成的函数模型写入该数据库；vast-checker 后续读取这个数据库，使用其中的模型和已有 callee 结果完成分析，并将新的分析结果写回。函数模型与 checker 生成的结果按来源和版本区分，但保存在同一个数据库中。


## 2. 构建与分析入口

构建项目之前，将 C、C++ 编译器分别设置为：

```bash
export CC=wllvm
export CXX=wllvm++
```

`wllvm` 和 `wllvm++` 是编译器包装器：先正常调用 clang/gcc 编译，再调用 `vast-front` 生成 IR。构建环境同时配置好包装器调用前端所需的环境变量。CMake 项目使用新的构建目录重新配置，保证编译器设置生效。

构建完成后，通过 WLLVM4Vast 中的 `extract-bc` 链接生成的 IR，得到 `.thin` 文件。该文件记录符号定义位置、call graph、class hierarchy 和部分全局常量值。

checker 的分析入口为：

```bash
vast-checker -a <analysis-name> -t <thin-file>
```

例如：

```bash
vast-checker -a mld -t project.thin
vast-checker -a uaf -t project.thin
vast-checker -a df -t project.thin
```

每轮运行固定源码版本、构建配置、`.thin` 和模型版本，使建模、分析和验证使用同一份输入。


## 3. 函数摘要到底描述什么

摘要只保留与资源分析有关、能够跨越函数调用边界的效果。函数内部的计算过程和局部临时变量不需要完整搬进模型。

### 第一部分：分配了什么，保存到哪里

记录新创建的资源及其出口。例如：

- 创建一个 buffer，地址写入第一个参数所指对象的 `buffer` 字段。
- 创建一个对象，通过返回值交给调用方。
- 创建一个句柄，通过输出参数交给调用方。

这里的“哪里”可以是返回值、参数指向的位置、对象字段或调用方可访问的全局位置。

### 第二部分：释放了什么

记录哪个已有资源被释放，以及释放后哪些调用方可见的位置发生变化。例如：释放参数对象的 `buffer`，并将该字段设为空；或释放参数直接指向的对象。

释放动作与字段赋值分别记录：释放资源不自动意味着所有别名都被清空，把指针设为空也不等于释放资源。

### 第三部分：做了什么调用方可见的修改

记录会影响后续资源判断的修改，例如：把一个参数中的指针保存到对象字段、让返回值指向已有 buffer、将资源移入容器，或替换调用方持有的指针。

“调用方可见”包括通过参数可访问的对象、返回值、可达字段和全局状态。与当前资源规则无关的数值更新，可以从资源摘要中省略。

### 所有权：谁负责维持和结束资源生命周期

所有权贯穿上述三部分：

- **调用方拥有**：函数返回后，调用方负责释放资源。
- **对象拥有**：资源属于某个对象，由该对象的清理接口负责释放。
- **借用**：只访问已有资源，释放责任不改变。
- **转移**：释放责任从调用方转移给对象或另一个持有者。

所有权与指针指向关系分别记录。两个位置指向同一 buffer，不意味着它们都拥有 buffer；某个调用方不再拥有资源，也不意味着资源已经被释放。

摘要还记录行为成立的条件。例如，创建成功才产生资源，插入成功才转移所有权，失败时仍由原持有者负责释放。checker 根据调用结果把效果应用到对应分支。


## 4. 用 vector 理解建模结果

考虑一个拥有内部 buffer 的 `vector` 类。其接口可以总结为：

| 函数 | 分配 | 释放 | 调用方可见的资源修改 | 所有权 |
| --- | --- | --- | --- | --- |
| `vec_init(v)` | 创建 buffer，保存到 `v.buffer` | 无 | 建立 `v` 与 buffer 的关联 | vector 对象 `v` 拥有 buffer |
| `vec_push(v, value)` | 无 | 无 | 无需记录新的资源效果 | buffer 继续由 `v` 拥有 |
| `vec_free(v)` | 无 | 释放 `v.buffer` 指向的 buffer | 如实现会清空字段，则记录该修改 | 结束 `v` 对 buffer 的所有权 |

对于这组接口，checker 可以把调用理解为：

```text
vec_init(&v)
  → v 获得一个由它拥有的 buffer

vec_push(&v, value)
  → 资源及所有权状态保持不变

vec_free(&v)
  → v 拥有的 buffer 被释放
```

`vec_push` 的“什么都没做”指**对当前资源分析没有需要记录的新效果**。元素写入、长度更新等行为仍然会发生，只是不需要在这个资源摘要中展开。

如果某个实现的 `vec_push` 会扩容，Agent 会在摘要中记录扩容分支：分配新 buffer、更新 `v.buffer`、释放旧 buffer，且新 buffer 仍由 `v` 拥有。如果插入操作会接管元素指向的资源，还要记录元素所有权转移。摘要以实际实现为依据选择这些效果。

这样，调用者可以调用多次 `vec_push`，checker 无须每次展开容器内部实现；它只需跟踪 vector 持有的资源是否按接口约定完成生命周期。


## 5. Agent 如何为库函数建模

### 选择目标

建模服务优先处理频繁调用的库接口、资源管理封装和容器接口。同一实现的摘要可以被多个调用点复用，减少 checker 重复分析。

### 收集上下文

服务根据 `.thin` 中的符号定义位置和调用关系，读取目标函数、相关类型、必要的被调用函数及代表性调用点。函数名中的 create、init、free、destroy 等关键词用于筛选候选；真正的行为由实现和接口约定确定。

### 生成摘要

Agent 按“分配、释放、调用方可见修改、所有权”提取效果，同时记录成功与失败条件、资源关联位置及源码依据。输出参数、返回值和对象字段使用统一的访问路径表示，便于 checker 映射到调用点。

### 检查与入库

模型管理器检查函数标识、参数与字段路径、条件表达，以及所有权关系的一致性。无法确定或无法表达的关键效果保留为未知，函数继续交给 checker 分析，不将缺失效果解释为空效果。

检查通过后，plugin 将摘要写入分析数据库，并记录目标函数、适用版本和模型来源。vast-checker 后续使用这个数据库分析：查询到适用模型时，直接应用摘要，不再展开该函数实现。模型写入与 checker 分析可以分步执行，后续运行仍可复用数据库中已有的模型。源码或相关依赖变化时，模型失效并重新建模，依赖它的分析结果也重新计算。

数据库中的每个模型条目包含：目标函数、适用实现版本、资源效果、所有权关系、成立条件、源码证据和模型版本。


## 6. Agent 如何为函数指针建模

函数指针建模同时处理**调用目标**和**调用效果**。

首先，Agent 从函数指针的赋值、注册接口、结构体字段、初始化表和调用点中识别可能的目标。类型信息与 class hierarchy 辅助缩小范围；关键词不能替代目标解析。

随后，服务为候选目标建立摘要，并把目标集合与对应效果绑定到间接调用点。例如：

```text
对象的 destroy 回调
  可能调用 buffer_destroy 或 vector_destroy
          ↓
分别建立两个目标的资源摘要
          ↓
checker 分别应用候选效果，再合并可能的资源状态
```

当目标能够唯一确定时，应用该目标摘要；存在多个候选时，保留各候选的可能效果。只有所有候选都保证发生的释放，才能作为确定释放传播。仍存在未知目标时保留未知分支，不能把已识别的候选当作完整集合。

对于 callback 参数，模型记录回调绑定方式、传入的上下文对象，以及回调是否被保存供以后调用。注册回调不等于立即执行回调；资源是否被接管、何时释放，需要按注册接口和实际回调的语义分别表达。

这使 checker 可以使用已整理的目标和资源效果，减少对库内部间接调用的重复推导，同时保留目标不确定性。


## 7. checker 如何使用模型完成分析

vast-checker 根据 call graph 和符号定义位置，从叶子函数逐步分析到 `main`。分析结果写入一个类似 `.git` 的数据库，caller 可以查询 callee 的分析结果。

处理调用时分为三种情况：

1. **存在函数模型**：读取分析数据库中的模型，直接应用摘要，将参数、返回值和字段映射到当前调用者的实际对象。
2. **没有模型但有实现**：分析函数实现，保存结果供 caller 查询。
3. **间接调用**：使用候选目标及其摘要或分析结果，合并可能行为；未知目标保留未知效果。

应用摘要时，checker 更新资源身份、持有位置、释放状态和所有权。例如，`vec_init(&v)` 建立 `v` 拥有 buffer 的关系；`vec_free(&v)` 结束该资源生命周期。随后沿调用者控制流检查是否存在未释放资源、重复释放或释放后访问。

vast-checker 的报错信息包含以下三部分：

```text
[bug-class] position
allocation stack
error path
```

- **`[bug-class] position`**：错误类别和报错位置，帮助 Agent 确定检查目标，并定位到对应源码。
- **`allocation stack`**：资源分配时的调用栈，帮助 Agent 追溯资源在哪里创建、经过哪些调用，以及最初由谁持有。
- **`error path`**：导致错误的程序路径，帮助 Agent 沿相关分支和调用检查资源如何被传递、修改、释放或使用。

Agent 以这些报错信息为入口，再按需查询分析数据库中的函数模型和 callee 结果，定位并验证错误。模型更新时，通过依赖关系重新分析受影响的 caller，避免使用旧摘要得出的结果。


## 8. Agent 如何验证分析结果

验证服务以 checker 输出的 `[bug-class] position`、`allocation stack` 和 `error path` 为入口，结合源码与分析数据库，验证 vast-checker 的报错是否成立。验证阶段的目标是判定 checker 报告的问题是否真实存在。

具体验证步骤为：

1. **定位报错位置**：根据 `bug-class` 确定检查的问题类型，根据 `position` 打开对应源码，确认触发错误的操作。
2. **追溯资源来源**：沿 `allocation stack` 查看分配点和调用链，确认资源身份、保存位置及初始所有权。
3. **逐段检查错误路径**：沿 `error path` 对照源码，检查分支条件、函数调用、别名变化、所有权转移和释放操作，判断路径上的资源状态是否支持该错误。
4. **检查反证**：检查是否已有有效释放、所有权转移，或是否存在使错误路径不可达的条件，确认报错是否与实际代码行为一致。
5. **给出验证结论**：指出支持错误的关键位置和路径；如果有反证或信息缺口，明确记录，询问用户，获得必要信息后继续验证。

以 vector 为例：

- `vec_init` 成功后直接返回，且 vector 没有转交给其他对象：检查是否遗漏 `vec_free`，造成内部 buffer 泄漏。
- 调用两次 `vec_free`：检查接口是否允许重复清理、第一次是否清空字段，以及第二次是否真的释放同一资源。
- `vec_free` 后继续通过旧指针访问 buffer：检查别名关系和路径条件，确认是否构成释放后使用。
- checker 在 `vec_push` 后报告错误：结合实际扩容、指针替换和元素所有权行为，判断所报告的资源错误是否真实存在。

复核结论分为：

| 结论 | 含义 | 动作 |
| --- | --- | --- |
| 确认问题 | 源码与资源路径支持 checker 的报错 | 写入 tp.json，说明成立依据 |
| 确认误报 | 源码证据表明 checker 所报错误不存在 | 写入 fp.json，说明反证 |
| 信息不足 | 关键实现或路径证据缺失 | 询问用户，获得信息后继续验证 |

验证完成后，将 checker 的真实报错与误报分别写入下述结果文件，并保留判断依据。

### 验证结果输出

所有分类均以 **vast-checker 是否报错**为基准，再根据 Agent 的验证结论判断该报告或未报告是否正确。当前阶段验证 checker 已报告的告警，输出两份 JSON 文件，均以数组保存结果：

- **`tp.json`**：记录 vast-checker 报告且经 Agent 验证确实存在的错误（True Positive）。
- **`fp.json`**：记录 vast-checker 的误报（False Positive），即 checker 报告了错误，但 Agent 验证后确认该错误实际不存在。

两份文件中的每条记录使用相同字段：

| 字段 | 含义 |
| --- | --- |
| `bug-class` | checker 报告的错误类别 |
| `position` | checker 报告的错误位置 |
| `path` | 对应的 error path，保留关键位置、调用和资源操作 |
| `confidence` | Agent 对本条分类结论的置信度，取值为 0～1；在 `fp.json` 中表示对“vast-checker 误报了”这一判断的信心 |
| `description` | 验证理由，结合分配栈、错误路径、源码及函数模型，说明问题为何成立或为何是误报 |

`tp.json` 示例：

```json
[
  {
    "bug-class": "uaf",
    "position": "src/example.c:42",
    "path": [
      "src/example.c:20: vec_init 创建 buffer",
      "src/example.c:35: vec_free 释放 buffer",
      "src/example.c:42: 通过旧指针访问已释放的 buffer"
    ],
    "confidence": 0.95,
    "description": "分配栈确认旧指针来自该 vector 的 buffer。错误路径上 buffer 已被释放，后续访问前没有重新分配，源码与函数模型均支持该释放后使用问题。"
  }
]
```

`fp.json` 示例：

```json
[
  {
    "bug-class": "mld",
    "position": "src/example.c:60",
    "path": [
      "src/example.c:20: vec_init 创建 buffer",
      "src/example.c:50: cleanup 清理 vector",
      "src/example.c:60: 函数返回"
    ],
    "confidence": 0.92,
    "description": "检查错误路径及 cleanup 的实现，确认函数返回前已调用 vec_free 释放该 buffer，因此 vast-checker 报告的泄漏实际不存在，判定为误报。"
  }
]
```

示例中的路径与数值仅用于展示记录结构。`confidence` 表示 Agent 的判断信心，不等于经过统计校准的正确率。证据不足的告警保留为待验证，不强行写入 TP 或 FP；没有对应结果时，文件内容为 `[]`。


## 9. 后续演进：单函数双路分析

下一步为 checker 增加单函数分析入口，让 subagent 与 checker 对同一函数分别分析，再对比结果。

```mermaid
flowchart LR
    A["同一函数与上下文"] --> B["subagent 独立分析"]
    A --> C["checker 单函数分析"]
    B --> D["对比并验证"]
    C --> D
    D --> E["TP / FP / TN / FN"]
    D -->|证据不足| F["询问用户"]
    F --> D
```

subagent 先独立输出结果，再读取 checker 的结果。checker 在这一模式下展开目标函数实现，绕过该函数已有的替代模型，避免两边实际上使用同一份摘要。两边使用一致的 callee 上下文，并统一参数、字段路径及资源标识。

比较重点是：是否遗漏资源、释放对象是否一致、调用方可见修改是否完整、所有权是否正确、条件是否匹配。结果一致仍保留依据；存在差异时，根据源码和路径证据定位原因。

单函数比较完成后，记录 checker 正确报告、误报及漏报的问题和验证依据。函数建模负责为库函数与函数指针提供摘要；结果验证负责判断 checker 的分析与报错是否正确。

### 扩展验证范围：TN 与 FN

当前验证以 vast-checker 已报告的告警为入口，因此输出 TP 和 FP。单函数双路分析进一步覆盖 checker 没有报错的代码：subagent 独立检查目标函数的资源行为，再与 checker 的结果比较，可以补充 TN 和 FN。

四种分类始终相对于 vast-checker：

| vast-checker 的判断 | Agent 验证后的结论 | 分类 | 结果文件 |
| --- | --- | --- | --- |
| 报错 | 对应错误确实存在 | TP：checker 正确报告 | `tp.json` |
| 报错 | 对应错误实际不存在 | FP：checker 误报 | `fp.json` |
| 未报错 | 检查范围内不存在对应错误 | TN：checker 正确未报告 | `tn.json` |
| 未报错 | 对应错误实际存在 | FN：checker 漏报 | `fn.json` |

分类以具体错误类型、位置和路径对应的问题为单位。同一函数可能同时包含正确报告的问题和另一个被漏报的问题，不能只给整个函数贴一个标签。

TN 和 FN 的验证流程为：

1. 固定目标函数、错误类型、分析上下文和检查范围。
2. subagent 独立检查资源分配、释放、调用方可见修改和所有权，给出源码与路径依据。
3. 对齐 checker 已报告的问题，识别未被报告的候选错误。
4. 对未报告的候选进一步验证：确认存在则记录为 FN；对明确检查过且确认不存在对应错误的范围，记录为 TN；证据不足则保留待验证。
5. 对 FN 记录真实错误的位置、触发路径，以及 checker 未报告该问题的对比证据。

`tn.json`、`fn.json` 沿用 `bug-class`、`position`、`path`、`confidence`、`description` 字段。由于 checker 未报错，这些记录的位置与路径由 Agent 检查产生；TN 的 `description` 明确限定检查范围和成立条件，FN 的 `description` 说明真实错误及 checker 遗漏的证据。

四类记录中的 `confidence` 都表示 Agent 对相应分类结论的信心。checker 没有报错本身不能证明 TN，Agent 与 checker 的结论不一致也不能直接证明 FN；分类需要对应的验证依据。


## 10. 用验证结果改善 vast-checker 的分析精度

TP、FP、TN、FN 形成持续改进 checker 的反馈闭环。当前阶段通过 TP/FP 评估报错质量；后续单函数双路分析补充 TN/FN，进一步发现漏报。

```mermaid
flowchart LR
    A["TP / FP / TN / FN"] --> B["提取复现案例"]
    B --> C["改进 checker"]
    C --> D["回归验证精度"]
    D --> E["接受改进"]
    D -->|未改善| C
    E --> A
```

| 验证结果 | 对 checker 改进的作用 |
| --- | --- |
| TP：正确报错 | 保留为缺陷检测用例，确保后续修改仍能检出 |
| FP：误报 | 提取反证和触发路径，改进 checker 的判断，减少无效报错 |
| TN：正确未报错 | 保留为正常用例，防止改进检测能力时引入误报 |
| FN：漏报 | 提取真实错误和触发条件，补足 checker 的检测能力 |

改进对象是 **vast-checker 的分析与报错判断**。Agent 提供分类和证据，开发者据此定位分析逻辑问题，完成修改后用同一组案例验证效果。评估时同时关注误报与漏报，不能仅靠减少报错数量判断精度提升。

对已验证的结果，可以统计报错准确率 `TP / (TP + FP)`；后续补充 FN 后，可统计检出率 `TP / (TP + FN)`。指标在固定检查范围、问题单位和验证标准下比较，证据不足的条目不纳入已确认分类。
